# 02: 量子通信におけるノイズ限界と安全な鍵生成率
### (Noise Limits and Secret Key Rates in Quantum Communication)

---

## 1. Thesis: 科学的問いと仮説

量子鍵配送（QKD）は理論上、完全なセキュリティを保証しますが、現実のデバイスや光ファイバーには必ず「ノイズ」が存在します。ノイズによるエラー（QBER）と、盗聴（Eve）によるエラーは区別がつかないため、ある一定以上のノイズが発生すると、安全な鍵を抽出できなくなります。

**問い** : 量子チャンネルのノイズ（誤り率）が増大したとき、最終的な「安全な鍵の生成率（Secret Key Rate）」はどのように減衰し、どの程度の誤り率が通信の限界となるのか？

**仮説** : アリスとボブが共有する情報量（相互情報量）から、盗聴者イヴが得る可能性のある情報の最大値（レベノ・バウンドに類する限界）を差し引いた値が正である必要がある。BB84プロトコルにおいて、単純なエラー訂正とプライバシー増幅を用いた場合、QBERが約 $11\%$ を超えると安全な鍵の生成率がゼロになり、通信が成立しなくなる。


## 2. Theoretical Background (理論的背景)

### 2.1 相互情報量とデバタクの定理
安全な鍵生成率 $R$ は、理想的には以下のように表されます：
$$R \geq I(A:B) - I(A:E)$$
- $I(A:B)$: アリスとボブの間の相互情報量（通信路の質）
- $I(A:E)$: アリスとイヴの間の相互情報量（盗聴者に漏れた情報量）

### 2.2 セキュリティの臨界点 (11.0% limit)
BB84プロトコルにおいて、イヴが最も効率的な攻撃（最適攻撃）を行ったと仮定すると、ボブが観測する誤り率 $e$ に対して、エラー訂正に $H_2(e)$ ビット、プライバシー増幅にさらに $H_2(e)$ ビットのコストがかかると考えると、
$$R=1−2H_2(e)$$
という簡略化されたモデルで評価できます。この $R$ が $0$ になる点が、BB84の理論的限界である **約 11.0%** です。


## 3. Implementation (実装)

量子チャンネルに意図的に制御されたノイズ（ビット反転エラー）を注入し、さまざまなノイズレベル（0% 〜 20%）において、アリスとボブの鍵がどれだけ一致し、理論上の「安全な鍵生成率」がどう変化するかをシミュレートします。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

def binary_entropy(p):
    """バイナリエントロピー関数 H2(p)"""
    if p <= 0 or p >= 1:
        return 0
    return -p * np.log2(p) - (1-p) * np.log2(1-p)

def simulate_noise_effect(noise_rate):
    """指定されたノイズ率（QBER）における鍵生成の挙動をシミュレートします。"""
    num_bits = 1000
    # 実際には通信を行わず、統計的な結果をシミュレート
    # (QBER=noise_rateの状態を作成)
    alice_bits = np.random.randint(2, size=num_bits)
    errors = np.random.random(num_bits) < noise_rate
    bob_bits = np.array([b if not e else 1-b for b, e in zip(alice_bits, errors)])
    
    # 相互情報量の算出 (1 - H2(e))
    # 理論的な安全な鍵生成率 R = 1 - 2*H2(e) 
    # (簡略化モデル: エラー訂正とプライバシー増幅のコスト)
    h2 = binary_entropy(noise_rate)
    key_rate = max(0, 1 - 2 * h2)
    
    actual_qber = np.mean(alice_bits != bob_bits)
    return actual_qber, key_rate

# さまざまなノイズレベルで実験
noise_levels = np.linspace(0, 0.20, 25)
qber_results = []
key_rate_results = []

for nl in noise_levels:
    q, r = simulate_noise_effect(nl)
    qber_results.append(q)
    key_rate_results.append(r)

print("シミュレーション完了。解析結果を出力します。")


## 4. Visualization & Analysis (可視化と解析)

「ノイズ（QBER）」と「最終的な安全な鍵生成率」の関係をプロットします。
QBERが増大するにつれて、安全に抽出できるビットの割合がどのように急落するかを確認します。


In [ ]:
plt.figure(figsize=(9, 6))

# キーレートのプロット
plt.plot(qber_results, key_rate_results, 'o-', color='#1f77b4', label='Secret Key Rate $R$')

# 11% の限界線を引く
plt.axvline(x=0.11, color='#d62728', linestyle='--', label='Theoretical Limit (~11%)')
plt.fill_between(qber_results, 0, key_rate_results, where=(np.array(qber_results) <= 0.11), color='#1f77b4', alpha=0.2)

plt.title("Security Limit Analysis: Secret Key Rate vs. QBER")
plt.xlabel("Quantum Bit Error Rate (QBER)")
plt.ylabel("Secret Key Rate (bits per pulse)")
plt.grid(True, which="both", ls="-", alpha=0.4)
plt.legend()

# 注釈
plt.annotate('Shor-Preskill Limit', xy=(0.11, 0.05), xytext=(0.14, 0.2),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=8))

plt.show()


## 5. Conclusion & Future Work (結論と展望)

### 結論
分析結果から、量子通信の実用上の限界が浮き彫りになりました：

1. **ノイズの隠蔽性** : QBERが $11\%$ 付近に近づくと、安全な鍵生成率は急速にゼロに収束します。これは、ノイズが「盗聴者の情報量」と区別できない以上、一定以上のエラーがある通信路では、プライバシー増幅を行っても秘密を守り切れない（秘密情報のビットが残らない）ことを意味します。

2. **実用的な閾値** : 通信システムを設計する際、QBERを $11\%$ 未満、できれば十分に余裕を持った数パーセント以下に抑えるハードウェア要件が必須であることが確認されました。

### 展望

- **高効率なエラー訂正符号（LDPC codes）** : 現在の簡略化モデル（$2H_2$）よりも効率的なエラー訂正アルゴリズムを用いることで、限界値をわずかに押し上げたり、同じノイズレベルでもより多くの鍵を抽出したりすることが可能です。

- **デバイス非確実性への対応** : 実際の光学系に含まれる「サイドチャネル」からの情報漏洩を考慮した、より厳格なセキュリティ解析モデルの実装が、商用QKDシステムには求められます。
